# FastSLAM 1.0

## 1. Configuração da simulação
- Imports
- Parâmetros
- Ambiente
- Landmarks
- Robô
- Sensor

## 2. Trajetória
- Definição da trajetória
- Visualização dos comandos
- \(v\) comandado × aplicado
- \(\omega\) comandado × aplicado

## 3. Ground Truth × Perception
- Simulação do robô
- Aquisição das medições
- Animação Ground Truth
- Animação Perception

## 4. Modelo de movimento
- Configuração dos ruídos
- Experimento do modelo de movimento
- Visualização das trajetórias amostradas

## 5. Modelo de observação
- Visualização das medições
- Range
- Bearing
- Ground Truth × medição

## 6. Data Association
- Observações
- Predições das medições
- Likelihoods
- Associação ML
- Comparação com correspondência verdadeira

## 7. FastSLAM 1.0
- Configuração do número de partículas
- Execução do algoritmo
- Animação das partículas
- Evolução dos mapas

## 8. Resampling
- Pesos das partículas
- Distribuição dos pesos
- Evolução do número de partículas efetivas
- Visualização do resampling

## 9. Resultados
- Trajetória Ground Truth × FastSLAM
- Estimativa dos landmarks
- Erro de posição
- Erro de orientação
- Erro dos landmarks
- Incertezas

## 10. Avaliação da associação
- Correspondência estimada × Ground Truth
- Taxa de acerto
- Associações incorretas
- Casos de outlier

## 11. Experimentos
- Diferentes números de partículas
- Diferentes níveis de ruído
- Influência da frequência de medição
- Comparação dos resultados

## 12. Comparação com EKF-SLAM
- Trajetórias
- Erros
- Mapas
- Associação
- Desempenho

## 13. Discussão
- Análise dos resultados
- Comportamento observado
- Limitações
- Casos de falha

## 14. Conclusão
- Principais resultados do experimento

In [1]:
%load_ext autoreload
%autoreload 2

In [29]:
import sys
import numpy as np
import matplotlib.pyplot as plt
import os

sys.path.append(os.path.abspath(r"../util/"))

from geometry import Pose, Angle
from maps import Landmark, LandmarkMap
from camera import Camera, DetectedFeature
from robot import Robot
from control import VelocityControl
from fastslam import FastSLAM, LandmarkEstimate

In [10]:
# Teste matemático do modelo de observação
pose = Pose(0, 0, 0)

landmark_mu = np.array([
    [3.0],
    [4.0]
])

z_hat, delta, q = FastSLAM.predict_measurement(
    pose,
    landmark_mu
)

print("delta:")
print(delta)

print("\nq:")
print(q)

print("\nz_hat:")
print(z_hat)

print("\nrange esperado:", 5.0)
print("bearing esperado:", np.arctan2(4, 3))

delta:
[[3.]
 [4.]]

q:
25.0

z_hat:
[[5.        ]
 [0.92729522]]

range esperado: 5.0
bearing esperado: 0.9272952180016122


In [11]:
# Teste 2 — Jacobiano H

H = FastSLAM.measurement_jacobian(
    delta,
    q
)

print("H calculado:")
print(H)

H_expected = np.array([
    [0.6,  0.8],
    [-0.16, 0.12]
])

print("\nH esperado:")
print(H_expected)

print("\nErro:")
print(H - H_expected)

print("\nJacobiano correto?")
print(np.allclose(H, H_expected))

H calculado:
[[ 0.6   0.8 ]
 [-0.16  0.12]]

H esperado:
[[ 0.6   0.8 ]
 [-0.16  0.12]]

Erro:
[[0. 0.]
 [0. 0.]]

Jacobiano correto?
True


In [13]:
# Teste 3 — Inicialização do landmark
print("\n\nTeste 3 — Inicialização do landmark")
pose = Pose(0, 0, 0)

landmark_mu = np.array([
    [3.0],
    [4.0]
])

z = DetectedFeature(
    r=5.0,
    phi=Angle(np.arctan2(4, 3)),
    signature=0
)

Q = np.diag([
    0.05**2,
    np.deg2rad(1.0)**2
])

landmark_est = FastSLAM.initialize_landmark(
    pose,
    z,
    Q
)

print("mu estimado:")
print(landmark_est.mu)

print("\nmu esperado:")
print(landmark_mu)

print("\nErro:")
print(landmark_est.mu - landmark_mu)

print("\nSigma:")
print(landmark_est.sigma)

print("\nSigma simétrica?")
print(np.allclose(
    landmark_est.sigma,
    landmark_est.sigma.T
))



Teste 3 — Inicialização do landmark
mu estimado:
[[3.]
 [4.]]

mu esperado:
[[3.]
 [4.]]

Erro:
[[ 4.4408921e-16]
 [-4.4408921e-16]]

Sigma:
[[ 0.00577388 -0.00245541]
 [-0.00245541  0.00434156]]

Sigma simétrica?
True


In [14]:
# Teste 4 — EKF do landmark
print("\n\nTeste 4 — EKF do landmark")

pose = Pose(0, 0, 0)

Q = np.diag([
    0.05**2,
    np.deg2rad(1.0)**2
])

z1 = DetectedFeature(
    r=5.0,
    phi=Angle(np.arctan2(4, 3)),
    signature=0
)

landmark = FastSLAM.initialize_landmark(
    pose,
    z1,
    Q
)


z2 = DetectedFeature(
    r=5.1,
    phi=Angle(np.arctan2(4.1, 3.0)),
    signature=0
)

fs = FastSLAM(
    n_particles=1,
    q_mat=Q
)
mu_before = landmark.mu.copy()
sigma_before = landmark.sigma.copy()
diag = fs.update_landmark(
    landmark,
    pose,
    z2
)

print("=== ANTES ===")
print("mu:")
print(mu_before)

print("\nSigma:")
print(sigma_before)

print("\n=== DEPOIS ===")
print("mu:")
print(landmark.mu)

print("\nSigma:")
print(landmark.sigma)

print("\n=== INOVAÇÃO ===")
print(diag["innovation"])

print("\n=== KALMAN GAIN ===")
print(diag["K"])



Teste 4 — EKF do landmark
=== ANTES ===
mu:
[[3.]
 [4.]]

Sigma:
[[ 0.00577388 -0.00245541]
 [-0.00245541  0.00434156]]

=== DEPOIS ===
mu:
[[3.00637905]
 [4.05771571]]

Sigma:
[[ 0.00288694 -0.0012277 ]
 [-0.0012277   0.00217078]]

=== INOVAÇÃO ===
[[0.1       ]
 [0.01181047]]

=== KALMAN GAIN ===
[[ 0.3 -2. ]
 [ 0.4  1.5]]


In [ ]:
# Teste 5 — Associação de landmarks
print("\n\nTeste 5 — Associação de landmarks")
fs = FastSLAM(
    n_particles=1,
    q_mat=Q,
    p0=1e-3
)

fs.initialize(Pose(0, 0, 0))

particle = fs.particles[0]

z_l0 = DetectedFeature(
    r=3.0,
    phi=Angle(0.0),
    signature=0
)

z_l1 = DetectedFeature(
    r=4.0,
    phi=Angle(np.pi/2),
    signature=1
)

particle.landmarks[0] = FastSLAM.initialize_landmark(
    particle.pose, z_l0, Q
)

particle.landmarks[1] = FastSLAM.initialize_landmark(
    particle.pose, z_l1, Q
)

z_test = DetectedFeature(
    r=3.0,
    phi=Angle(0.0),
    signature=0
)

correspondence, likelihood, diag = fs.associate(
    z_test,
    particle
)

print("Correspondência escolhida:", correspondence)
print("Likelihood escolhida:", likelihood)

print("\nLikelihood de cada hipótese:")
for idx, value in diag["likelihoods"].items():
    print(f"Landmark {idx}: {value}")

Correspondência escolhida: 0
Likelihood escolhida: 91.18906527810405

Likelihood de cada hipótese:
Landmark 0: 91.18906527810405
Landmark 1: 5e-324
Landmark 2: 0.001


In [19]:
# Teste 6 — verificar criação de novo landmark
print("\n\nTeste 6 — verificar criação de novo landmark")

z_new = DetectedFeature(
    r=5.0,
    phi=Angle(np.pi),
    signature=99
)

correspondence, likelihood, diag = fs.associate(
    z_new,
    particle
)

print("Correspondência escolhida:", correspondence)
print("Likelihood escolhida:", likelihood)

print("\nLikelihood de cada hipótese:")
for idx, value in diag["likelihoods"].items():
    print(f"Landmark {idx}: {value}")



Teste 6 — verificar criação de novo landmark
Correspondência escolhida: None
Likelihood escolhida: 0.001

Likelihood de cada hipótese:
Landmark 0: 5e-324
Landmark 1: 5e-324
Landmark 2: 0.001


In [22]:
fs = FastSLAM(
    n_particles=1,
    q_mat=Q,
    p0=1e-3
)

pose0 = Pose(0, 0, 0)
fs.initialize(pose0)

# Landmark ainda não conhecido
z = DetectedFeature(
    r=3.0,
    phi=Angle(0.0),
    signature=0
)

motion = VelocityControl(
    v=0.0,
    w=0.0,
    dt=0.1
)

diagnostics = fs.update(
    motion=motion,
    detected_features=[z],
    motion_noise=False,
    resample=False
)

particle = fs.particles[0]

print("=== PARTÍCULA ===")
print("Pose:", particle.pose)
print("Peso:", particle.weight)

print("\n=== LANDMARKS ===")
for idx, landmark in particle.landmarks.items():
    print(f"Landmark {idx}:")
    print("  mu =", landmark.mu.ravel())
    print("  sigma =")
    print(landmark.sigma)

print("\n=== DIAGNÓSTICOS ===")
print("Pesos:", diagnostics["weights_before_resampling"])
print("N_eff:", diagnostics["effective_particle_number"])

=== PARTÍCULA ===
Pose: Pose(0.00, 0.00, Angle(0.00°))
Peso: 0.0010000000000000002

=== LANDMARKS ===
Landmark 0:
  mu = [3. 0.]
  sigma =
[[0.0025     0.        ]
 [0.         0.00274156]]

=== DIAGNÓSTICOS ===
Pesos: [0.001]
N_eff: 1.0


In [23]:
# Teste 8 — atualização de um landmark já existente
print("\n\nTeste 8 — atualização de um landmark já existente")

fs = FastSLAM(
    n_particles=1,
    q_mat=Q,
    p0=1e-3
)

fs.initialize(Pose(0, 0, 0))

# Primeira observação: cria o landmark
z1 = DetectedFeature(
    r=3.0,
    phi=Angle(0.0),
    signature=0
)

motion = VelocityControl(
    v=0.0,
    w=0.0,
    dt=0.1
)

fs.update(
    motion=motion,
    detected_features=[z1],
    motion_noise=False,
    resample=False
)

landmark_before = fs.particles[0].landmarks[0]

mu_before = landmark_before.mu.copy()
sigma_before = landmark_before.sigma.copy()

# Segunda observação, ligeiramente diferente
z2 = DetectedFeature(
    r=3.1,
    phi=Angle(np.deg2rad(1.0)),
    signature=0
)

diagnostics = fs.update(
    motion=motion,
    detected_features=[z2],
    motion_noise=False,
    resample=False
)

landmark_after = fs.particles[0].landmarks[0]

print("=== ANTES DA SEGUNDA OBSERVAÇÃO ===")
print("mu:")
print(mu_before)

print("sigma:")
print(sigma_before)

print("\n=== DEPOIS DA SEGUNDA OBSERVAÇÃO ===")
print("mu:")
print(landmark_after.mu)

print("sigma:")
print(landmark_after.sigma)

print("\n=== DIAGNÓSTICOS ===")
print("Peso:", fs.particles[0].weight)
print("N_eff:", diagnostics["effective_particle_number"])



Teste 8 — atualização de um landmark já existente
=== ANTES DA SEGUNDA OBSERVAÇÃO ===
mu:
[[3.]
 [0.]]
sigma:
[[0.0025     0.        ]
 [0.         0.00274156]]

=== DEPOIS DA SEGUNDA OBSERVAÇÃO ===
mu:
[[3.05      ]
 [0.02617994]]
sigma:
[[0.00125    0.        ]
 [0.         0.00137078]]

=== DIAGNÓSTICOS ===
Peso: 26.12610462337377
N_eff: 1.0


In [24]:
# Teste 9 — várias partículas e pesos
print("\n\nTeste 9 — várias partículas e pesos")

fs = FastSLAM(
    n_particles=20,
    q_mat=Q,
    p0=1e-3
)

fs.initialize(Pose(0, 0, 0))

# Criamos um landmark conhecido em cada partícula
z1 = DetectedFeature(
    r=3.0,
    phi=Angle(0.0),
    signature=0
)

motion = VelocityControl(
    v=0.0,
    w=0.0,
    dt=0.1
)

fs.update(
    motion=motion,
    detected_features=[z1],
    motion_noise=False,
    resample=False
)

# Segunda observação
z2 = DetectedFeature(
    r=3.1,
    phi=Angle(np.deg2rad(1.0)),
    signature=0
)

diagnostics = fs.update(
    motion=motion,
    detected_features=[z2],
    motion_noise=False,
    resample=False
)

print("=== PESOS ===")
print(diagnostics["weights_before_resampling"])

print("\n=== PESOS NORMALIZADOS ===")
print(diagnostics["normalized_weights"])

print("\nSoma dos pesos normalizados:",
      np.sum(diagnostics["normalized_weights"]))

print("\nN_eff:",
      diagnostics["effective_particle_number"])



Teste 9 — várias partículas e pesos
=== PESOS ===
[26.12610462 26.12610462 26.12610462 26.12610462 26.12610462 26.12610462
 26.12610462 26.12610462 26.12610462 26.12610462 26.12610462 26.12610462
 26.12610462 26.12610462 26.12610462 26.12610462 26.12610462 26.12610462
 26.12610462 26.12610462]

=== PESOS NORMALIZADOS ===
[0.05 0.05 0.05 0.05 0.05 0.05 0.05 0.05 0.05 0.05 0.05 0.05 0.05 0.05
 0.05 0.05 0.05 0.05 0.05 0.05]

Soma dos pesos normalizados: 1.0000000000000002

N_eff: 19.99999999999999


In [25]:
# Teste 10 — gerar diversidade entre partículas
print("\n\nTeste 10 — gerar diversidade entre partículas")
fs = FastSLAM(
    n_particles=20,
    q_mat=Q,
    p0=1e-3
)

fs.initialize(Pose(0, 0, 0))

z1 = DetectedFeature(
    r=3.0,
    phi=Angle(0.0),
    signature=0
)

motion = VelocityControl(
    v=1.0,
    w=0.0,
    dt=0.1
)

diagnostics = fs.update(
    motion=motion,
    detected_features=[z1],
    motion_noise=True,
    resample=False
)

print("=== POSES DAS PARTÍCULAS ===")

for i, particle in enumerate(fs.particles):
    print(
        f"Particle {i:02d}: "
        f"x={particle.pose.x:.4f}, "
        f"y={particle.pose.y:.4f}, "
        f"theta={particle.pose.th.rad:.4f}"
    )

print("\n=== PESOS NORMALIZADOS ===")

for i, weight in enumerate(
    diagnostics["normalized_weights"]
):
    print(f"Particle {i:02d}: {weight:.6f}")

print("\nN_eff:",
      diagnostics["effective_particle_number"])



Teste 10 — gerar diversidade entre partículas
=== POSES DAS PARTÍCULAS ===
Particle 00: x=0.0895, y=-0.0016, theta=-0.0365
Particle 01: x=0.1385, y=0.0004, theta=0.0059
Particle 02: x=0.1013, y=0.0003, theta=0.0061
Particle 03: x=0.1026, y=-0.0006, theta=-0.0121
Particle 04: x=0.0983, y=0.0002, theta=0.0039
Particle 05: x=0.0835, y=0.0001, theta=0.0026
Particle 06: x=0.0901, y=0.0002, theta=0.0042
Particle 07: x=0.0967, y=0.0003, theta=0.0070
Particle 08: x=0.1247, y=0.0008, theta=0.0120
Particle 09: x=0.1162, y=0.0001, theta=0.0023
Particle 10: x=0.1126, y=0.0008, theta=0.0136
Particle 11: x=0.0919, y=0.0001, theta=0.0019
Particle 12: x=0.0922, y=0.0005, theta=0.0117
Particle 13: x=0.0988, y=-0.0003, theta=-0.0068
Particle 14: x=0.1103, y=0.0003, theta=0.0050
Particle 15: x=0.1033, y=-0.0000, theta=-0.0000
Particle 16: x=0.0831, y=0.0006, theta=0.0149
Particle 17: x=0.0897, y=-0.0004, theta=-0.0090
Particle 18: x=0.0929, y=0.0004, theta=0.0076
Particle 19: x=0.1037, y=0.0003, theta=

In [30]:
# Teste 11 — partículas com pesos diferentes
print("\n\nTeste 11 — partículas com pesos diferentes")

# Vamos criar uma diferença artificial entre as hipóteses das partículas.
# Todas terão o mesmo landmark no mapa, mas poses diferentes.

for i, particle in enumerate(fs.particles):

    particle.landmarks.clear()

    landmark = LandmarkEstimate(
        mu=np.array([[3.0], [0.0]]),
        sigma=np.diag([0.01, 0.01]),
    )

    particle.landmarks[0] = landmark


# Observação verdadeira do landmark (3, 0)
z_test = DetectedFeature(
    r=3.0,
    phi=Angle(0.0),
    signature=0
)


print("=== LIKELIHOOD POR PARTÍCULA ===")

likelihoods = []

for i, particle in enumerate(fs.particles):

    likelihood, z_hat, psi = fs.measurement_likelihood(
        z_test,
        particle.landmarks[0],
        particle.pose,
    )

    likelihoods.append(likelihood)

    print(
        f"Particle {i:02d}: "
        f"pose=({particle.pose.x:.4f}, "
        f"{particle.pose.y:.4f}, "
        f"{particle.pose.th.rad:.4f}) "
        f"| likelihood={likelihood:.8f}"
    )



Teste 11 — partículas com pesos diferentes
=== LIKELIHOOD POR PARTÍCULA ===
Particle 00: pose=(0.0895, -0.0016, -0.0365) | likelihood=16.87377480
Particle 01: pose=(0.1385, 0.0004, 0.0059) | likelihood=16.71007814
Particle 02: pose=(0.1013, 0.0003, 0.0061) | likelihood=24.11617380
Particle 03: pose=(0.1026, -0.0006, -0.0121) | likelihood=22.94759320
Particle 04: pose=(0.0983, 0.0002, 0.0039) | likelihood=24.91029296
Particle 05: pose=(0.0835, 0.0001, 0.0026) | likelihood=27.93401871
Particle 06: pose=(0.0901, 0.0002, 0.0042) | likelihood=26.52731049
Particle 07: pose=(0.0967, 0.0003, 0.0070) | likelihood=24.93032379
Particle 08: pose=(0.1247, 0.0008, 0.0120) | likelihood=18.68985648
Particle 09: pose=(0.1162, 0.0001, 0.0023) | likelihood=21.33410613
Particle 10: pose=(0.1126, 0.0008, 0.0136) | likelihood=20.73323179
Particle 11: pose=(0.0919, 0.0001, 0.0019) | likelihood=26.28878946
Particle 12: pose=(0.0922, 0.0005, 0.0117) | likelihood=25.06703281
Particle 13: pose=(0.0988, -0.0003

In [31]:
weights = np.array(likelihoods)

# Normalizar
weights = weights / np.sum(weights)

print("=== PESOS NORMALIZADOS ===")
for i, w in enumerate(weights):
    print(f"Particle {i:02d}: {w:.6f}")

print("\nSoma:", np.sum(weights))

print("\nN_eff:", 1.0 / np.sum(weights**2))

=== PESOS NORMALIZADOS ===
Particle 00: 0.035957
Particle 01: 0.035608
Particle 02: 0.051389
Particle 03: 0.048899
Particle 04: 0.053082
Particle 05: 0.059525
Particle 06: 0.056527
Particle 07: 0.053124
Particle 08: 0.039826
Particle 09: 0.045461
Particle 10: 0.044181
Particle 11: 0.056019
Particle 12: 0.053416
Particle 13: 0.052301
Particle 14: 0.047695
Particle 15: 0.051169
Particle 16: 0.055390
Particle 17: 0.055459
Particle 18: 0.054547
Particle 19: 0.050425

Soma: 1.0000000000000002

N_eff: 19.662731943815007


In [32]:
# Teste 12 - Resampling
print("\n\nTeste 12 - Resampling")
# Teste 12 — Resampling

# Usar os likelihoods obtidos no Teste 11
weights = np.array(likelihoods, dtype=float)

# Colocar os pesos diretamente nas partículas
for particle, weight in zip(fs.particles, weights):
    particle.weight = weight

print("=== PESOS ANTES DO RESAMPLING ===")

normalized_before = fs.normalized_weights()

for i, weight in enumerate(normalized_before):
    print(f"Particle {i:02d}: {weight:.6f}")

print("\nN_eff antes:",
      fs.effective_particle_number())


# Fixar a semente para tornar o experimento reproduzível
np.random.seed(42)

# Resampling
indices = fs.resample()

print("\n=== ÍNDICES SELECIONADOS ===")
print(indices)

print("\n=== QUANTIDADE DE VEZES QUE CADA PARTÍCULA FOI SELECIONADA ===")

counts = np.bincount(
    indices,
    minlength=fs.n_particles
)

for i, count in enumerate(counts):
    print(f"Particle {i:02d}: {count} vez(es)")


print("\n=== PESOS DEPOIS DO RESAMPLING ===")
print([particle.weight for particle in fs.particles])



Teste 12 - Resampling
=== PESOS ANTES DO RESAMPLING ===
Particle 00: 0.035957
Particle 01: 0.035608
Particle 02: 0.051389
Particle 03: 0.048899
Particle 04: 0.053082
Particle 05: 0.059525
Particle 06: 0.056527
Particle 07: 0.053124
Particle 08: 0.039826
Particle 09: 0.045461
Particle 10: 0.044181
Particle 11: 0.056019
Particle 12: 0.053416
Particle 13: 0.052301
Particle 14: 0.047695
Particle 15: 0.051169
Particle 16: 0.055390
Particle 17: 0.055459
Particle 18: 0.054547
Particle 19: 0.050425

N_eff antes: 19.662731943815007

=== ÍNDICES SELECIONADOS ===
[ 7 19 14 12  3  3  1 17 12 14  0 19 16  4  4  4  6 11  8  6]

=== QUANTIDADE DE VEZES QUE CADA PARTÍCULA FOI SELECIONADA ===
Particle 00: 1 vez(es)
Particle 01: 1 vez(es)
Particle 02: 0 vez(es)
Particle 03: 2 vez(es)
Particle 04: 3 vez(es)
Particle 05: 0 vez(es)
Particle 06: 2 vez(es)
Particle 07: 1 vez(es)
Particle 08: 1 vez(es)
Particle 09: 0 vez(es)
Particle 10: 0 vez(es)
Particle 11: 1 vez(es)
Particle 12: 2 vez(es)
Particle 13: 0

In [ ]:
import numpy as np

# ============================================================
# TESTE 13 — FastSLAM em trajetória simples
# ============================================================

# Mapa verdadeiro
landmarks_gt = {
    0: np.array([5.0, 2.0]),
    1: np.array([5.0, 5.0]),
    2: np.array([0.0, 5.0]),
}

# FastSLAM
fs = FastSLAM(
    n_particles=10,
    q_mat=np.diag([
        0.05**2,
        np.deg2rad(1.0)**2
    ]),
    p0=1e-3,
)

# Pose inicial
true_pose = Pose(0.0, 0.0, 0.0)

fs.initialize(true_pose)

# Controle
dt = 1.0
motion = VelocityControl(
    v=1.0,
    w=0.0,
    dt=dt,
)

# ============================================================
# Função para gerar observações a partir da pose verdadeira
# ============================================================

def generate_observations(pose, landmarks):
    features = []

    for signature, lm in landmarks.items():

        dx = lm[0] - pose.x
        dy = lm[1] - pose.y

        r = np.sqrt(dx**2 + dy**2)

        phi = np.arctan2(dy, dx) - float(pose.th.rad)

        # Normaliza para [-pi, pi]
        phi = np.arctan2(np.sin(phi), np.cos(phi))

        features.append(
            DetectedFeature(
                r=r,
                phi=Angle(phi),
                signature=signature,
            )
        )

    return features

# ============================================================
# Simulação
# ============================================================

n_steps = 5

true_pose = Pose(0.0, 0.0, 0.0)

true_poses = [true_pose]

print("=== TESTE 13 ===")

for step in range(n_steps):

    # Movimento verdadeiro
    true_pose, u_eff = motion.applyControl(
        true_pose,
        motion_noise=False,
    )

    true_poses.append(true_pose)

    # Observações geradas pela pose verdadeira
    features = generate_observations(
        true_pose,
        landmarks_gt,
    )

    # FastSLAM
    result = fs.update(
        motion=motion,
        detected_features=features,
        motion_noise=False,
        resample=True,
    )

    print(f"\nStep {step + 1}")
    print(f"Pose verdadeira: {true_pose}")
    print(f"N_eff: {result['effective_particle_number']:.3f}")

=== TESTE 13 ===

Step 1
Pose verdadeira: Pose(1.00, 0.00, Angle(0.00°))
N_eff: 10.000

Step 2
Pose verdadeira: Pose(2.00, 0.00, Angle(0.00°))
N_eff: 10.000

Step 3
Pose verdadeira: Pose(3.00, 0.00, Angle(0.00°))
N_eff: 10.000

Step 4
Pose verdadeira: Pose(4.00, 0.00, Angle(0.00°))
N_eff: 10.000


ValueError: Cannot predict a bearing for a landmark at the robot pose.

In [38]:
pose_test = Pose(0.0, 0.0, 0.0)

result = motion.applyControl(
    pose_test,
    motion_noise=False,
)

print("Tipo do retorno:", type(result))
print("Retorno:", result)

Tipo do retorno: <class 'tuple'>
Retorno: (Pose(1.00, 0.00, Angle(0.00°)), <control.VelocityControl object at 0x0000022A6E8F6150>)
